# Empirical Results: Adaptive Boolean Oracle (Marena 2026)
**Catholic University of America**

This notebook generates all empirical figures for the paper:
1. Adaptive vs Uniform error vs sample count (blind-oracle model, two regimes)
2. Tightness sweep over (N, K) pairs
3. NISQ noise crossover
4. k-Forrelation quantum vs classical separation
5. Kernel vs linear shadow accuracy

**Model note:** All bounds in Cells 2-3 assume the *blind-oracle model*
(sampling distribution p is independent of f). See PhaseTimeBound.lean
for the formal scope. Under concentrated support (p zero off-supp(f)),
the adaptive bound can be beaten; see GapAnalysis.md.

**Cell 2 note:** Two panels are produced:
- **Left (main figure):** N=512, K=16 (N/K=32, sparsity 3.1%).
  Adaptive theory curve: sqrt(K*pi^2/M_main) with pilot_frac<=0.1.
  Crossover expected near M ~ K*pi^2/eps^2 ~ 1750 (for eps=0.3),
  well within the M=200..20000 simulation window.
- **Right (large-N panel):** N=1024, K=16 (N/K=64, sparsity 1.6%).
  Same crossover location (same K=16), shown for scaling comparison.

Runtime on Colab CPU: ~20 minutes total.

## Cell 1 — Install and clone

In [ ]:
import subprocess, sys, os, urllib.request, zipfile

if not os.path.exists('/content/quantum_oracle_sketching'):
    url = 'https://github.com/Tommaso-R-Marena/quantum_oracle_sketching/archive/refs/heads/main.zip'
    urllib.request.urlretrieve(url, '/content/repo.zip')
    with zipfile.ZipFile('/content/repo.zip', 'r') as z:
        z.extractall('/content/')
    os.rename('/content/quantum_oracle_sketching-main', '/content/quantum_oracle_sketching')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'jax', 'jaxlib', 'numpy', 'matplotlib',
    'scikit-learn', 'scipy', 'tqdm', 'pyqsp', 'hatchling'], check=True)

# --force-reinstall --no-deps ensures the updated source is always loaded,
# even if Colab has cached .pyc files from a previous session.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    '--force-reinstall', '--no-deps', '-e',
    '/content/quantum_oracle_sketching'], check=True)

if '/content/quantum_oracle_sketching/src' not in sys.path:
    sys.path.insert(0, '/content/quantum_oracle_sketching/src')

os.chdir('/content/quantum_oracle_sketching')

# Purge any stale cached modules so fresh source is used
import importlib
for mod in list(sys.modules.keys()):
    if mod.startswith('qos'):
        del sys.modules[mod]

from qos.theory.adaptive_lower_bound import (
    compute_bounds, tightness_sweep,
    uniform_vs_adaptive_error_comparison,
    adversarial_sparse_function,
    nisq_adaptive_crossover,
)
from qos.core.oracle_sketch import q_oracle_sketch_boolean, q_oracle_sketch_boolean_adaptive

import jax, jax.numpy as jnp
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams.update({
    'font.size': 13, 'axes.titlesize': 13, 'axes.labelsize': 12,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight'
})
os.makedirs('/content/quantum_oracle_sketching/results', exist_ok=True)

import qos
print(f'✓ Setup complete. QOS version: {qos.__version__}')

## Cell 2 — Figure 1: Adaptive vs Uniform Error (two regimes)
**Left panel (main figure):** N=512, K=16 — pilot_frac=min(0.1, 20K/M).
**Right panel (large-N):** N=1024, K=16 — same K, larger N for scaling comparison.

Theory curves (blind-oracle model, t=pi*K for adaptive):
  Uniform:  sqrt(N * pi^2 / M)
  Adaptive: sqrt(K * pi^2 / M_main)  where M_main = M*(1-pilot_frac) >= 0.9*M

Crossover expected near M ~ K*pi^2/eps^2 (adaptive converges).
For K=16, eps=0.3: crossover at M ~ 1750 (within simulation window).
Expected runtime: ~5 minutes.

In [ ]:
sample_counts = [200, 500, 1000, 2000, 5000, 10000, 20000]

# ── Panel A: N=512, K=16 (main figure) ──────────────────────────────────────
Na, Ka = 512, 16
print(f'Panel A: N={Na}, K={Ka}, N/K={Na//Ka}, (N/K)^2={(Na//Ka)**2}')
print(f'Adaptive crossover: M ~ K*pi^2/eps^2 = {int(Ka*3.14159**2/0.3**2)} (eps=0.3)')
results_a = uniform_vs_adaptive_error_comparison(
    N=Na, K=Ka, sample_counts=sample_counts, num_trials=20,
    key=jax.random.PRNGKey(0),
)
df1a = pd.DataFrame(results_a)
df1a.to_csv('results/fig1a_N512_K16.csv', index=False)
print(df1a.to_string(index=False))
print(f'N/K improvement factor: {Na/Ka:.0f}x (blind-oracle model)\n')

# ── Panel B: N=1024, K=16 (large-N regime) ──────────────────────────────────
Nb, Kb = 1024, 16
print(f'Panel B: N={Nb}, K={Kb}, N/K={Nb//Kb}, (N/K)^2={(Nb//Kb)**2}')
results_b = uniform_vs_adaptive_error_comparison(
    N=Nb, K=Kb, sample_counts=sample_counts, num_trials=20,
    key=jax.random.PRNGKey(1),
)
df1b = pd.DataFrame(results_b)
df1b.to_csv('results/fig1b_N1024_K16.csv', index=False)
print(df1b.to_string(index=False))
print(f'N/K improvement factor: {Nb/Kb:.0f}x (blind-oracle model)\n')

# ── Plot ─────────────────────────────────────────────────────────────────────
M_arr = np.array(sample_counts, float)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for ax, df, N, K, label in [
    (ax1, df1a, Na, Ka, '(a) N=512, K=16  [N/K=32, main figure]'),
    (ax2, df1b, Nb, Kb, '(b) N=1024, K=16  [N/K=64, large-N regime]'),
]:
    pilot_frac_arr = np.minimum(0.1, 20.0 * K / M_arr)
    M_main_arr = M_arr * (1 - pilot_frac_arr)
    theory_u = np.sqrt(N * np.pi**2 / M_arr)
    theory_a = np.sqrt(K * np.pi**2 / M_main_arr)
    ax.errorbar(df['M'], df['uniform_error'],  yerr=df['uniform_std'],
                fmt='o-', color='#d62728', label=f'Uniform QOS (N={N})', capsize=4)
    ax.errorbar(df['M'], df['adaptive_error'], yerr=df['adaptive_std'],
                fmt='s-', color='#1f77b4', label=f'Adaptive QOS (K={K})', capsize=4)
    ax.plot(M_arr, theory_u, 'r--', alpha=0.5, label=r'Theory: $\sqrt{N\pi^2/M}$')
    ax.plot(M_arr, theory_a, 'b--', alpha=0.5, label=r'Theory: $\sqrt{K\pi^2/M_{\rm main}}$')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('Sample count M')
    ax.set_ylabel(r'$L_\infty$ error on $\mathrm{supp}(f)$')
    ax.set_title(label)
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

fig.suptitle(
    'Adaptive vs Uniform Oracle Sketch Error\n'
    '(error on supp(f), blind-oracle model, 20 trials each)',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.savefig('results/fig1_combined.pdf')
plt.savefig('results/fig1_combined.png')
plt.show()

# Save individual panels for paper submission
for df, N, K, fname in [
    (df1a, Na, Ka, 'fig1a_main'),
    (df1b, Nb, Kb, 'fig1b_largeN'),
]:
    M_arr2 = np.array(sample_counts, float)
    pilot_frac_arr2 = np.minimum(0.1, 20.0 * K / M_arr2)
    M_main_arr2 = M_arr2 * (1 - pilot_frac_arr2)
    fig2, ax = plt.subplots(figsize=(6, 5))
    theory_u2 = np.sqrt(N * np.pi**2 / M_arr2)
    theory_a2 = np.sqrt(K * np.pi**2 / M_main_arr2)
    ax.errorbar(df['M'], df['uniform_error'],  yerr=df['uniform_std'],
                fmt='o-', color='#d62728', label=f'Uniform QOS (N={N})', capsize=4)
    ax.errorbar(df['M'], df['adaptive_error'], yerr=df['adaptive_std'],
                fmt='s-', color='#1f77b4', label=f'Adaptive QOS (K={K})', capsize=4)
    ax.plot(M_arr2, theory_u2, 'r--', alpha=0.5, label=r'Theory: $\sqrt{N\pi^2/M}$')
    ax.plot(M_arr2, theory_a2, 'b--', alpha=0.5, label=r'Theory: $\sqrt{K\pi^2/M_{\rm main}}$')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('Sample count M')
    ax.set_ylabel(r'$L_\infty$ error on $\mathrm{supp}(f)$')
    ax.set_title(
        f'Adaptive vs Uniform Oracle Sketch Error\n'
        f'(N={N}, K={K}, sparsity={K/N:.1%}, blind-oracle model)'
    )
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'results/{fname}.pdf')
    plt.savefig(f'results/{fname}.png')
    plt.close()
    print(f'Saved {fname}.pdf / .png')

## Cell 3 — Figure 2: Tightness Sweep over (N, K)
Uses explicit integer K values to avoid NaN in pivot.
M_adaptive = K*pi^2/eps^2  (adaptive phase time t=pi*K).
M_uniform  = N*pi^2/eps^2  (uniform phase time t=pi*N).
Expected runtime: ~1 minute.

In [ ]:
NK_pairs = [
    (128, 1), (128, 6), (128, 13), (128, 32),
    (256, 3), (256, 13), (256, 26), (256, 64),
    (512, 5), (512, 26), (512, 51), (512, 128),
    (1024, 10), (1024, 51), (1024, 102), (1024, 256),
]
epsilon = 0.1

rows = []
for N, K in NK_pairs:
    r = compute_bounds(N, K, epsilon)
    rows.append({
        'N': N, 'K': K,
        'sparsity_pct': f'{K/N:.0%}',
        'M_lower': r.M_lower_bound,
        'M_adaptive': r.M_adaptive_upper,
        'M_uniform': r.M_uniform_upper,
        'improvement_factor': r.improvement_factor,
        'tight': r.is_tight,
    })
df2 = pd.DataFrame(rows)
df2.to_csv('results/fig2_tightness_sweep.csv', index=False)
print(df2.to_string(index=False))

import numpy as np
heat_df = pd.DataFrame(rows)
pivot_data = heat_df.pivot_table(index='N', columns='K', values='improvement_factor', aggfunc='first')

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(pivot_data.values, aspect='auto', cmap='Blues')
ax.set_xticks(range(len(pivot_data.columns)))
ax.set_xticklabels([f'K={k}' for k in pivot_data.columns])
ax.set_yticks(range(len(pivot_data.index)))
ax.set_yticklabels(pivot_data.index)
ax.set_xlabel('Support size K'); ax.set_ylabel('N')
ax.set_title('Improvement Factor N/K (Adaptive vs Uniform, blind-oracle model)')
for i in range(pivot_data.values.shape[0]):
    for j in range(pivot_data.values.shape[1]):
        v = pivot_data.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.0f}x', ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=ax, label='N/K')
plt.savefig('results/fig2_tightness_sweep.pdf')
plt.savefig('results/fig2_tightness_sweep.png')
plt.show()

## Cell 4 — Figure 3: NISQ Noise Crossover
Expected runtime: ~30 seconds.

In [ ]:
from qos.primitives.noise_model import DepolarizingChannel, crossover_sample_count

N, K = 512, 16
noise_rates = np.logspace(-4, -1, 30)
circuit_depth = 50
epsilon_target = 0.3

rows = []
for p in noise_rates:
    r = nisq_adaptive_crossover(N, K, float(p), circuit_depth, epsilon_target)
    rows.append({
        'noise_rate': p,
        'epsilon_noise': r['epsilon_noise'],
        'epsilon_sketch_budget': r['epsilon_sketch_budget'],
        'M_adaptive': r['M_adaptive_crossover'],
        'M_uniform': r['M_uniform_crossover'],
        'beneficial': r['adaptive_still_beneficial'],
    })
df3 = pd.DataFrame(rows)
df3.to_csv('results/fig3_nisq_crossover.csv', index=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.loglog(df3['noise_rate'], df3['M_adaptive'], 'b-', label=f'Adaptive (K={K})')
ax1.loglog(df3['noise_rate'], df3['M_uniform'], 'r-', label=f'Uniform (N={N})')
ax1.axvline(x=epsilon_target/circuit_depth, color='gray', linestyle='--',
            label=r'$\varepsilon_{\rm noise} = \varepsilon_{\rm target}$')
ax1.set_xlabel('Per-gate noise rate p'); ax1.set_ylabel('Crossover sample count M*')
ax1.set_title('NISQ Crossover: M* vs Noise Rate')
ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.semilogx(df3['noise_rate'], df3['epsilon_sketch_budget'], 'g-')
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax2.set_xlabel('Per-gate noise rate p')
ax2.set_ylabel(r'Remaining budget $\varepsilon_{\rm sketch}$')
ax2.set_title(f'Sketch Budget After Noise (depth={circuit_depth})')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/fig3_nisq_crossover.pdf')
plt.savefig('results/fig3_nisq_crossover.png')
plt.show()

## Cell 5 — Figure 4: k-Forrelation Quantum vs Classical Separation
Expected runtime: ~3 minutes.

In [ ]:
try:
    from qos.data.generation import k_forrelation_data
    from qos.utils.numerical import unnormalized_hadamard_transform
    _forrelation_ok = True
except ImportError as e:
    print(f'Skipping Fig 4 (import error: {e})')
    _forrelation_ok = False

if _forrelation_ok:
    n_bits = 8
    k_values = [2, 3, 4, 5]
    num_trials = 10
    epsilon = 0.1
    rows = []
    for k in k_values:
        errors, classical_bounds = [], []
        for trial in range(num_trials):
            key = jax.random.PRNGKey(trial * 100 + k)
            gen = k_forrelation_data(n=n_bits, k=k, key=key)
            func_key = jax.random.PRNGKey(trial * 100 + k + 1)
            funcs = gen.sample_functions(func_key)
            exact = gen.compute_exact_forrelation(funcs)
            truth = ((1.0 - jnp.sign(funcs[-1])) / 2.0).astype(jnp.int32)
            diag, _ = q_oracle_sketch_boolean(truth, 5000)
            est = gen.quantum_query_algorithm(diag)
            errors.append(abs(float(est) - float(exact)))
            classical_bounds.append(gen.classical_streaming_complexity(epsilon))
        rows.append({
            'k': k,
            'mean_error': float(np.mean(errors)),
            'std_error': float(np.std(errors)),
            'classical_bound': int(np.mean(classical_bounds)),
        })
    df4 = pd.DataFrame(rows)
    df4.to_csv('results/fig4_k_forrelation.csv', index=False)
    print(df4.to_string(index=False))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.errorbar(df4['k'], df4['mean_error'], yerr=df4['std_error'],
                 fmt='o-', color='purple', capsize=5)
    ax1.set_xlabel('k'); ax1.set_ylabel('|estimated - exact|')
    ax1.set_title('QOS Estimator Error vs k-Forrelation Order')
    ax1.grid(True, alpha=0.3)
    ax2.semilogy(df4['k'], df4['classical_bound'], 's-', color='red',
                 label=r'Classical lower bound $\Omega(N^{1-1/k}/\varepsilon^2)$')
    ax2.set_xlabel('k'); ax2.set_ylabel('Sample complexity lower bound')
    ax2.set_title('Classical Streaming Lower Bound vs k')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/fig4_k_forrelation.pdf')
    plt.savefig('results/fig4_k_forrelation.png')
    plt.show()

## Cell 6 — Figure 5: Kernel vs Linear Shadow Accuracy
Expected runtime: ~2 minutes.

In [ ]:
try:
    from qos.core.state_sketch import (
        q_state_sketch_flat, fit_kernel_svm_from_states,
        q_interferometric_kernel_shadow,
    )
    _kernel_ok = True
except ImportError as e:
    print(f'Skipping Fig 5 (import error: {e})')
    _kernel_ok = False

if _kernel_ok:
    def run_shadow_comparison(dim, n_train, n_test, sketch_samples, key):
        k1, k2, _ = jax.random.split(key, 3)
        X_train = jax.random.normal(k1, (n_train, dim))
        y_train = jnp.sign(X_train[:, 0]).astype(jnp.int32)
        X_test = jax.random.normal(k2, (n_test, dim))
        y_test = jnp.sign(X_test[:, 0]).astype(jnp.int32)
        sketch_train = jnp.stack([q_state_sketch_flat(X_train[i], sketch_samples)[0] for i in range(n_train)])
        sketch_test = jnp.stack([q_state_sketch_flat(X_test[i], sketch_samples)[0] for i in range(n_test)])
        alpha = fit_kernel_svm_from_states(sketch_train, y_train)
        kernel_preds = jnp.array([
            q_interferometric_kernel_shadow(sketch_train, y_train, alpha, sketch_test[i])
            for i in range(n_test)
        ])
        overlaps = jnp.abs(jnp.einsum('id,jd->ij', sketch_test, sketch_train))**2
        linear_preds = y_train[jnp.argmax(overlaps, axis=1)]
        return float(jnp.mean(kernel_preds == y_test)), float(jnp.mean(linear_preds == y_test))

    dims = [16, 32, 64, 128]
    rows = []
    for d in dims:
        k_accs, l_accs = [], []
        for trial in range(8):
            ka, la = run_shadow_comparison(d, 20, 40, 2000, jax.random.PRNGKey(trial * 7 + d))
            k_accs.append(ka); l_accs.append(la)
        rows.append({'dim': d,
                     'kernel_acc': np.mean(k_accs), 'kernel_std': np.std(k_accs),
                     'linear_acc': np.mean(l_accs), 'linear_std': np.std(l_accs)})
    df5 = pd.DataFrame(rows)
    df5.to_csv('results/fig5_kernel_vs_linear.csv', index=False)
    print(df5.to_string(index=False))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(df5['dim'], df5['kernel_acc'], yerr=df5['kernel_std'],
                fmt='s-', color='#1f77b4', label='Kernel shadow', capsize=4)
    ax.errorbar(df5['dim'], df5['linear_acc'], yerr=df5['linear_std'],
                fmt='o-', color='#ff7f0e', label='Linear shadow', capsize=4)
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Perfect accuracy')
    ax.set_xlabel('State dimension d'); ax.set_ylabel('Classification accuracy')
    ax.set_title('Kernel vs Linear Interferometric Shadow')
    ax.set_ylim(0.4, 1.05); ax.legend(); ax.grid(True, alpha=0.3)
    plt.savefig('results/fig5_kernel_vs_linear.pdf')
    plt.savefig('results/fig5_kernel_vs_linear.png')
    plt.show()

## Cell 7 — Download all results

In [ ]:
import shutil
results_dir = '/content/quantum_oracle_sketching/results'
print('=== Results saved ===')
for fname in sorted(os.listdir(results_dir)):
    path = os.path.join(results_dir, fname)
    print(f'  {fname:45s} {os.path.getsize(path):>8,} bytes')
shutil.make_archive('/content/marena2026_results', 'zip', results_dir)
print('\n✓ Download marena2026_results.zip from the Files panel (folder icon, left sidebar).')